# Simulation 4
In Simulation 4, we introduced sadness as a negative emotion that emerges when outcomes are perceived as uncontrollable. We examine how anger/frustration and sadness vary as a function of perceived controllability. Using the same task structure as in Simulation 3, controllability is high in the first block and then declines in the second block. This decline follows a logistic function, allowing the researcher to tune both the midpoint and the rate of decay. The logistic form is chosen for pragmatic demonstration purposes and is not meant to imply a specific psychological inference mechanism."

| Parameter | Range      | Interpretation |
|-----------|------------|----------------|
| η         | [0, 1]     | Learning rate: how quickly value expectations update |
| γ         | [0, 1]     | Discount factor: weight given to future rewards |
| λ_A       | [0, 1]     | Affective inertia: higher values → slower emotion updates |
| α         | [0, 1]     | Relative weighting of prediction errors versus absolute rewards as affective inputs (1 … only RPE) |
| κ         | > 1        | Negativity bias: negative affective inputs are amplified relative to positive ones |
| λ_C       | [0, 1]     | Controllability learning rate: speed at which perceived controllability declines across trials in the uncontrollable block |
| midpoint  | [100, 200] | Trial number at which C reaches 50% (inflection point of the decay) |


In [1]:
n_iterations = 20

group_cols = ["lambda_A", "lambda_C", "midpoint", "C", "eta", "gamma", "alpha",
              "kappa", "Step"]

In [2]:
import model
import importlib
importlib.reload(model)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mesa.visualization import SolaraViz, SpaceRenderer, make_plot_component
from mesa import batch_run
import numpy as np
import gc
import sys


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.options.display.float_format = "{:.3f}".format

init_variables = {
    # Variables

    "V": 0,
    "M_A": 0,  # anger
    "M_S": 0,  # sadness

    # Parameters

    # Parameter for the exponential moving average of anger
    "lambda_A": np.unique(np.concatenate([
        np.linspace(0.6, 1, 10),  # parameter sweep
        [8/9]   # plot for masters thesis (keeping values consistent)
    ])),


    "lambda_C": np.linspace(0, 1, 5),
    "midpoint": [125, 175, 200],

    # Learning rate
    "eta": np.unique(np.concatenate([
        np.linspace(0, 1, 5),  # parameter sweep
        [1/9]   # plot for masters thesis (keeping values consistent)
    ])),

    # Discounting factor
    "gamma": np.linspace(0, 1, 8),

    # Balance between RPE and absolute rewards
    "alpha": np.unique(np.concatenate([
        np.linspace(0, 1, 5),  # parameter sweep
        [4/9]   # plot for masters thesis (keeping values consistent)
    ])),

    # Negativity bias
    "kappa": np.unique(np.concatenate([
        np.linspace(1, 5, 5),  # parameter sweep
        [17/9] # plot for masters thesis (keeping values consistent)
    ])),
}

# We batch-run all parameter combinations.
# Because the model is stochastic, we run each combination multiple times
# (e.g., five iterations) to estimate the expected behavior.
# Seeding works as follows: each iteration is assigned a unique seed,
# and that same seed is used for all parameter combinations within that iteration.
# This ensures reproducibility while allowing multiple stochastic replications
# per parameter set.
rng = np.random.default_rng(12345)
seed_values = rng.integers(0, sys.maxsize, size=(n_iterations,))

for iteration, seed_value in enumerate(seed_values):
    print(f"Starting iteration {iteration}...", end="", flush=True)

    results = batch_run(
        model.IrritabilityModel,
        parameters=init_variables,
        data_collection_period=1,  # collect data for every step
        number_processes=16,
        rng=seed_value,
        max_steps=200,
    )

    results_df = pd.DataFrame(results)
    results_df["iteration"] = iteration

    # Sorting allows for row-based aggregation later
    results_df = results_df.sort_values(group_cols, ignore_index=True)

    results_df.to_parquet(f"output/004_sadness_{iteration}.parquet")

    del results
    del results_df
    gc.collect()

    print(f"Starting iteration {iteration}... done.")

# Quick test of sorting
df7 = pd.read_parquet("output/004_sadness_7.parquet", columns=group_cols)
df14 = pd.read_parquet("output/004_sadness_14.parquet", columns=group_cols)
assert df7.equals(df14), "Rows do not align after sorting!"

del df7, df14
gc.collect()

Starting iteration 0...

  0%|          | 0/285120 [00:00<?, ?it/s]

Starting iteration 0... done.
Starting iteration 1...

  0%|          | 0/285120 [00:00<?, ?it/s]

Starting iteration 1... done.
Starting iteration 2...

...

  0%|          | 0/285120 [00:00<?, ?it/s]

Starting iteration 18... done.
Starting iteration 19...

  0%|          | 0/285120 [00:00<?, ?it/s]

Starting iteration 19... done.


0

## Summary (expectation and standard errors)

In [3]:
import numpy as np

value_cols = ["V", "M_A", "M_S"]

n_rows = len(pd.read_parquet("output/004_sadness_0.parquet"))
n = 0
mean = {col: np.zeros(n_rows) for col in value_cols}
M2 = {col: np.zeros(n_rows) for col in value_cols}

for i in range(n_iterations):
    path = f"output/004_sadness_{i}.parquet"

    print(f"Processing iteration {i}...")

    df = pd.read_parquet(path, columns=value_cols)
    n += 1
    for c in value_cols:
        x = df[c].to_numpy()
        delta = x - mean[c]
        mean[c] += delta / n
        M2[c] += delta * (x - mean[c])

# final standard deviation
std = {c: np.sqrt(M2[c] / (n - 1)) for c in value_cols}

df = pd.read_parquet(
    "output/004_sadness_0.parquet",
    columns=group_cols
)

summary = df.copy()

for c in value_cols:
    summary[f"{c}_mean"] = mean[c]
    summary[f"{c}_std"] = std[c]

summary.to_parquet("output/004_sadness_summary.parquet")

Processing iteration 0...
Processing iteration 1...
Processing iteration 2...
Processing iteration 3...
Processing iteration 4...
Processing iteration 5...
Processing iteration 6...
Processing iteration 7...
Processing iteration 8...
Processing iteration 9...
Processing iteration 10...
Processing iteration 11...
Processing iteration 12...
Processing iteration 13...
Processing iteration 14...
Processing iteration 15...
Processing iteration 16...
Processing iteration 17...
Processing iteration 18...
Processing iteration 19...


# Downcast data for dashboard

In [4]:
import pandas as pd
import gc

# Summary dataframe

summary = pd.read_parquet("output/004_sadness_summary.parquet")

summary.info(memory_usage="deep")

for c in summary.select_dtypes(include="integer"):
    summary[c] = pd.to_numeric(summary[c], downcast="integer")

for c in summary.select_dtypes(include="float"):
    summary[c] = pd.to_numeric(summary[c], downcast="float")

summary.info(memory_usage="deep")

summary.to_parquet("output/dashboard/004_sadness_summary.parquet")

# Write meta-info
lambda_A_vals = sorted(summary["lambda_A"].unique())
lambda_C_vals = sorted(summary["lambda_C"].unique())
midpoint_vals = sorted(summary["midpoint"].unique())
eta_vals      = sorted(summary["eta"].unique())
gamma_vals    = sorted(summary["gamma"].unique())
C_vals        = sorted(summary["C"].unique())
alpha_vals        = sorted(summary["alpha"].unique())
kappa_vals        = sorted(summary["kappa"].unique())

meta_df = pd.DataFrame({
    "n_iterations": [n_iterations],
    "lambda_A_vals": [lambda_A_vals],
    "eta_vals": [eta_vals],
    "gamma_vals": [gamma_vals],
    "C_vals": [C_vals],
    "alpha_vals": [alpha_vals],
    "kappa_vals": [kappa_vals],
    "lambda_C_vals": [lambda_C_vals],
    "midpoint_vals": [midpoint_vals],
})

# Write to parquet
meta_df.to_parquet("output/dashboard/meta_info.parquet",
                   engine="pyarrow")

del summary
gc.collect()

# Individual iterations
n_iterations_dashboard = 3  # not all iterations to save memory

for i in range(n_iterations_dashboard):
    print("-------------------------------------------------------------------")

    df = pd.read_parquet(f"output/004_sadness_{i}.parquet")

    df.info(memory_usage="deep")

    df.drop(columns=["AgentID", "RunId", "iteration", "r", "rpe", "seed"],
            inplace=True)

    for c in df.select_dtypes(include="integer"):
        df[c] = pd.to_numeric(df[c], downcast="integer")

    for c in df.select_dtypes(include="float"):
        df[c] = pd.to_numeric(df[c], downcast="float")

    df.info(memory_usage="deep")

    df.to_parquet(f"output/dashboard/004_sadness_{i}.parquet")

    del df
    gc.collect()


<class 'pandas.DataFrame'>
RangeIndex: 57309120 entries, 0 to 57309119
Data columns (total 15 columns):
 #   Column    Dtype  
---  ------    -----  
 0   lambda_A  float64
 1   lambda_C  float64
 2   midpoint  int64  
 3   C         float64
 4   eta       float64
 5   gamma     float64
 6   alpha     float64
 7   kappa     float64
 8   Step      int64  
 9   V_mean    float64
 10  V_std     float64
 11  M_A_mean  float64
 12  M_A_std   float64
 13  M_S_mean  float64
 14  M_S_std   float64
dtypes: float64(13), int64(2)
memory usage: 6.4 GB
<class 'pandas.DataFrame'>
RangeIndex: 57309120 entries, 0 to 57309119
Data columns (total 15 columns):
 #   Column    Dtype  
---  ------    -----  
 0   lambda_A  float32
 1   lambda_C  float32
 2   midpoint  int16  
 3   C         float32
 4   eta       float32
 5   gamma     float32
 6   alpha     float32
 7   kappa     float32
 8   Step      int16  
 9   V_mean    float32
 10  V_std     float32
 11  M_A_mean  float32
 12  M_A_std   float32
 13  